# Variance Partitioning with scplainer

Quantifies how much of the variability in single-cell precursor intensities is explained by technical factors (normalisation, plate effects, injection position) versus biological factors (cell type, cell size). The analysis answers three questions:

1. **What fraction of per-precursor variance is attributable to each covariate?** The full model decomposes variability across normalisation, batch, and biological terms simultaneously.
2. **How does the variance structure change as technical effects are removed?** Sequential batch-effect regression reveals whether biological signal is masked by technical noise.
3. **Do cells separate by biology after batch correction?** GMF-based dimensionality reduction visualises the latent structure remaining at each correction stage.

**Key methodological choice** — scplainer fits a linear model per precursor via `scpModelWorkflow`. Because the model is feature-wise, the n/p ratio (observations per parameter) determines reliability of the models; precursors with n/p < 10 are excluded from downstream variance and dimensionality-reduction steps. Four correction stages are compared: raw (log2), post-normalisation regression, post-plate regression, and post-full-batch regression.

**Inputs** (`Data/AnnData/out/`)

| File | Description |
|---|---|
| `adata_precursor.h5ad` | Precursor-level quantification; X_log2 layer matrix used for modelling |

**Outputs** (`Data/scplainer/SC/precursor/`)

| File | Description |
|---|---|
| `sce.rds` | Full model SCE (all covariates) |
| `sce_norm.rds` | SCE after normalisation-term regression |
| `sce_plate.rds` | SCE after normalisation + plate regression |
| `sce_batch.rds` | SCE after full batch regression |
| `sce_resid.rds` | SCE residuals (all modelled effects removed) |
| `sce_GMF.rds` | GMF-reduced full SCE |
| `sce_norm_GMF.rds` | GMF-reduced post-normalisation SCE |
| `sce_plate_GMF.rds` | GMF-reduced post-plate SCE |
| `sce_batch_GMF.rds` | GMF-reduced post-batch SCE |
| `sce_batch.h5ad` | Batch-corrected SCE exported to AnnData format |

**Outputs** (`Figures/Figure_6/`)

| Subfolder | File pattern | Description |
|---|---|---|
| `Variance/` | `variance_bar_{stage}` | Average variance partitioning bar chart per correction stage |
| `Variance/` | `variance_violin_{stage}` | Per-precursor variance distribution violin plot per correction stage |

---

## Imports

In [2]:
library(reticulate)
use_condaenv("r45_scp", required = TRUE)
py_config()
library(anndata)

pkgs <- c(
    "anndata",
    "scp",
    "QFeatures",
    "SingleCellExperiment",
    "scater",
    "sgdGMF",
    "omicsGMF",
    "zellkonverter",
    "BiocParallel",
    "ggplot2",
    "patchwork",
    "dplyr",
    "gridExtra",
    "RColorBrewer",
    "scales"
)

cat("Package check\n")
cat(strrep("─", 40), "\n")
for (p in pkgs) {
    status  <- requireNamespace(p, quietly = TRUE)
    version <- if (status) as.character(packageVersion(p)) else "MISSING"
    cat(sprintf("  %-25s %s\n", p, version))
}
cat("\n")

suppressPackageStartupMessages({
    invisible(lapply(pkgs, library, character.only = TRUE))
})

# Parallel backend — MulticoreParam on Linux/Mac, SnowParam on Windows
library(BiocParallel)
if (.Platform$OS.type == "windows") {
    register(SnowParam(workers = 64))
} else {
    register(MulticoreParam(workers = 64))
}

python:         /home/sam/miniconda3/envs/r45_scp/bin/python
libpython:      /home/sam/miniconda3/envs/r45_scp/lib/libpython3.14.so
pythonhome:     /home/sam/miniconda3/envs/r45_scp:/home/sam/miniconda3/envs/r45_scp
version:        3.14.4 | packaged by conda-forge | (main, Apr  8 2026, 01:59:35) [GCC 14.3.0]
numpy:          /home/sam/miniconda3/envs/r45_scp/lib/python3.14/site-packages/numpy
numpy_version:  2.4.3

NOTE: Python version was forced by use_python() function

Package check
──────────────────────────────────────── 
  anndata                   0.9.0
  scp                       1.20.0
  QFeatures                 1.20.0
  SingleCellExperiment      1.32.0
  scater                    1.38.0
  sgdGMF                    1.0.1
  omicsGMF                  1.0.0
  zellkonverter             1.20.1
  BiocParallel              1.44.0
  ggplot2                   4.0.2
  patchwork                 1.3.2
  dplyr                     1.2.1
  gridExtra                 2.3
  RColorBrewer              1.1.3
  scales                    1.4.0



## 0. Path Configuration

Output directories are resolved relative to the notebook location and created automatically if absent. Both can be overridden via environment variables for CI or cross-machine runs.

| Variable | Default | Environment override |
|---|---|---|
| `ROOT_OUT` | `../../Data` | `SCI_DATA_DIR` |
| `FIGURES_OUT` | `../../Figures` | `SCI_FIGURES_DIR` |

Model RDS files are written to `ROOT_OUT/scplainer/SC/precursor/` and read back from there in later sections. AnnData objects are read from `ROOT_OUT/AnnData/out/`, produced by the QC pipeline.

In [3]:
NOTEBOOK_DIR <- normalizePath(getwd())
ROOT_OUT     <- normalizePath(file.path(NOTEBOOK_DIR, "../../Data"))
ROOT_OUT     <- Sys.getenv("SCI_DATA_DIR",    unset = ROOT_OUT)

FIGURES_OUT  <- normalizePath(file.path(NOTEBOOK_DIR, "../../Figures"))
FIGURES_OUT  <- Sys.getenv("SCI_FIGURES_DIR", unset = FIGURES_OUT)

ANNDATA_IN   <- file.path(ROOT_OUT, "AnnData", "out")
MODELS_OUT   <- file.path(ROOT_OUT, "scplainer", "SC", "precursor")

for (d in c(ROOT_OUT, FIGURES_OUT, ANNDATA_IN, MODELS_OUT,
            file.path(FIGURES_OUT, "Figure_6", "Variance"))) {
    dir.create(d, recursive = TRUE, showWarnings = FALSE)
}

cat(sprintf("Data directory    : %s\n", ROOT_OUT))
cat(sprintf("Figures directory : %s\n", FIGURES_OUT))
cat(sprintf("Models directory  : %s\n", MODELS_OUT))
cat(sprintf("AnnData input     : %s\n", ANNDATA_IN))

Data directory    : /home/sam/SCI/SCI/Data
Figures directory : /home/sam/SCI/SCI/Figures
Models directory  : /home/sam/SCI/SCI/Data/scplainer/SC/precursor
AnnData input     : /home/sam/SCI/SCI/Data/AnnData/out


## 1. Load Data

Load the precursor-level AnnData object and convert it into a `QFeatures` container for use with the `scp` modelling workflow. The log-intensity matrix is used.

The `colData` of the resulting `QFeatures` object carries all sample-level metadata needed for modelling: normalisation proxies (`medianIntensityPeptide`, `nPrecursors`), batch covariates (`plate_id`, `comment.X.position.`), and biological covariates (`source.name`, `Diameter`).

In [10]:
adata_prec <- read_h5ad(file.path(ANNDATA_IN, "adata_precursor.h5ad"))

int_mat   <- t(as.matrix(adata_prec$layers['X_log2']))
cell_meta <- adata_prec$obs

sce <- SingleCellExperiment(
    assays  = list(raw = int_mat),
    colData = DataFrame(cell_meta),
    rowData = DataFrame(adata_prec$var)
)

qf <- QFeatures(
    experiments = list(precursors_log = sce),
    colData     = DataFrame(cell_meta)
)

cat(sprintf("QFeatures: %d precursors x %d cells\n",
            nrow(qf[["precursors_log"]]), ncol(qf[["precursors_log"]])))
cat("\ncolData columns:\n")
print(names(colData(qf)))

QFeatures: 72061 precursors x 3114 cells

colData columns:
 [1] "sample_name"                     "plate_id"                       
 [3] "plate_idx"                       "source.name"                    
 [5] "comment.X.position."             "comment.Y.position."            
 [7] "comment.Plate.Number."           "comment.Gradient.time."         
 [9] "comment.thawing.timepoint."      "comment.plate.arrival."         
[11] "injection_order"                 "institute_name"                 
[13] "Reinjection"                     "Diameter"                       
[15] "Elongation"                      "Circularity"                    
[17] "Doublet"                         "Dividing"                       
[19] "Dying"                           "CellInBlank"                    
[21] "comment.plate.arrival._date"     "comment.plate.arrival._time"    
[23] "comment.plate.arrival._datetime" "comment.Gradient.time._date"    
[25] "comment.Gradient.time._time"     "comment.Gradient.time._da

## 2. Pre-processing

Three preparatory steps transform the raw intensity matrix into a form suitable for linear modelling:

1. **Log₂-transformation** — compresses the dynamic range and stabilises variance, making the linearity assumption of the linear model more appropriate (performed in previous python notebooks)
2. **Per-sample quality metrics** — `medianIntensityPeptide`, `n_precursors` (done), and `medianCV` are computed and stored in `colData`; these enter the model as normalisation covariates.
3. **Missing value audit** — the `Diameter` covariate (cell size from imaging) has ~11% missingness. Cells without a `Diameter` measurement are excluded when constructing the modelling dataset in Section 3.1. The full `qf` object — including these cells — is kept for QC visualisation and quality-metric computation above.

### 2.1 Log-transform and Quality Metrics

In [11]:
# Per-sample normalisation proxies
sample_medians          <- colMedians(assay(qf, "precursors_log"), na.rm = TRUE)
qf$medianIntensityPeptide <- sample_medians

# Median coefficient of variation per cell (grouped by protein)
qf <- medianCVperCell(
    qf,
    groupBy     = "PG.ProteinGroups",
    i           = "precursors_log",
    nobs        = 5,
    colDataName = "medianCV"
)

cat(sprintf("Added to colData: medianIntensityPeptide, nPrecursors, medianCV\n"))
summary(qf$medianIntensityPeptide)

Warning message in medianCVperCell(qf, groupBy = "PG.ProteinGroups", i = "precursors_log", :
“The median CV could not be computed for one or more samples. You may want to try a smaller value for 'nobs'.
”


Added to colData: medianIntensityPeptide, nPrecursors, medianCV


    Min.  1st Qu.   Median     Mean  3rd Qu.     Max.     NA's 
 0.04192  4.81342  5.29613  5.84352  6.48421 14.84085        3 

### 2.2 QC Visualisation

Six scatter plots (median intensity vs. precursor count) coloured by different metadata dimensions reveal whether any systematic structure exists before modelling. These are inspection plots only; cell filtering has already been performed upstream.

| Panel | Colour | What to look for |
|---|---|---|
| 1 | `medianCV` | High-CV cells may be low-quality or doublets |
| 2 | `source.name` | Cell types should occupy overlapping regions |
| 3 | `comment.X.position.` | Column position should not drive intensity |
| 4 | `plate_id` | Plate effects visible as intensity offsets |
| 5 | `Reinjection` | Re-injected cells should not cluster separately |
| 6 | `instrument_type` | Platform differences visible as horizontal bands |

In [12]:
cd <- data.frame(colData(qf))

base_aes <- aes(x = nPrecursors, y = medianIntensityPeptide)

htmp_1 <- ggplot(cd) + base_aes + geom_point(aes(color = medianCV), size = 2) +
    scale_color_continuous(type = "viridis") + labs(title = "medianCV")
htmp_2 <- ggplot(cd) + base_aes + geom_point(aes(color = source.name), size = 2) +
    labs(title = "Cell type")
htmp_3 <- ggplot(cd) + base_aes + geom_point(aes(color = factor(comment.X.position.)), size = 2) +
    labs(title = "Column position")
htmp_4 <- ggplot(cd) + base_aes + geom_point(aes(color = plate_id), size = 2) +
    labs(title = "Plate")
htmp_5 <- ggplot(cd) + base_aes + geom_point(aes(color = Reinjection), size = 2) +
    labs(title = "Reinjection")
htmp_6 <- ggplot(cd) + base_aes + geom_point(aes(color = instrument_type), size = 2) +
    labs(title = "Instrument")

options(repr.plot.width = 18, repr.plot.height = 10)
grid.arrange(htmp_1, htmp_2, htmp_3, htmp_4, htmp_5, htmp_6, ncol = 3, nrow = 2)

ERROR: [1m[33mError[39m in `geom_point()`:[22m
[1m[22m[33m![39m Problem while computing aesthetics.
[36mℹ[39m Error occurred in the 1st layer.
[1mCaused by error:[22m
[33m![39m object 'nPrecursors' not found


### 2.3 Missing Value Audit

Checks all formula variables for missing values before modelling. `Diameter` (cell size from fluorescence imaging) is the only variable with structural missingness (~11%), caused by cells that were not successfully imaged. Rather than mean-imputing — which would attenuate the estimated `Diameter` effect and bias the variance decomposition toward zero for that covariate — cells without a `Diameter` measurement are excluded when the modelling dataset is constructed in Section 3.1. All cells are retained here for QC inspection.

In [ ]:
formula_vars <- c("medianIntensityPeptide", "nPrecursors", "plate_id",
                  "comment.X.position.", "instrument_type", "source.name", "Diameter")

na_summary <- sapply(formula_vars, function(v) {
    x <- colData(qf)[[v]]
    c(n_NA    = sum(is.na(x)),
      pct_NA  = round(mean(is.na(x)) * 100, 1),
      n_total = length(x))
})

print(t(na_summary))

## 3. Modelling

Fits four scplainer models that correspond to successive correction stages. Each stage regresses out a subset of the full covariate set, and the residuals are passed to the next model. This design lets variance partitioning plots show how much variance each group of effects explains *on top of* the effects already removed.

| Stage | Model formula | What it captures |
|---|---|---|
| `sce` | All effects | Full decomposition: normalisation + batch + biology |
| `sce_norm` | − normalisation terms | Batch + biology after normalisation regression |
| `sce_plate` | − normalisation + plate | Column position + biology after plate correction |
| `sce_batch` | − normalisation + batch | Biology only after full batch regression |
| `sce_resid` | (residuals only) | Unexplained variance after removing all modelled effects |

<br>

> **Runtime note** — fitting the full model takes approximately 90–120 minutes on 64 cores. The RDS files are written after each fit so sections below can be run independently by loading the saved objects.

### 3.1 Fit Models

In [ ]:
# Exclude cells with missing Diameter — required for the linear model formula.
# qf (all cells) is kept intact above for QC; qf_model is used from here onward.
complete_cells <- !is.na(colData(qf)$Diameter)
qf_model <- qf[, complete_cells]

cat(sprintf("Cells with Diameter measurement : %d / %d  (%.1f%% retained)\n",
            sum(complete_cells), length(complete_cells),
            100 * mean(complete_cells)))

sce <- getWithColData(qf_model, "precursors_log")

In [ ]:
# --- Full model: all effects ---
cat("[1/5] Fitting full model (all effects)...\n")
sce <- scpModelWorkflow(
    sce,
    formula = ~ 1 +
        medianIntensityPeptide + nPrecursors +  # normalisation
        plate_id + comment.X.position. +        # batch
        source.name + Diameter                  # biology
)
saveRDS(sce, file.path(MODELS_OUT, "sce.rds"))
cat("  Saved: sce.rds\n\n")

# --- Regress out normalisation, refit ---
cat("[2/5] Regressing out normalisation terms...\n")
sce_norm <- scpRemoveBatchEffect(
    sce,
    effects   = c("medianIntensityPeptide", "nPrecursors"),
    intercept = TRUE
)
sce_norm <- scpModelWorkflow(
    sce_norm,
    formula = ~ 1 +
        plate_id + comment.X.position. +
        source.name + Diameter
)
saveRDS(sce_norm, file.path(MODELS_OUT, "sce_norm.rds"))
cat("  Saved: sce_norm.rds\n\n")

# --- Regress out normalisation + plate, refit ---
cat("[3/5] Regressing out normalisation + plate effects...\n")
sce_plate <- scpRemoveBatchEffect(
    sce,
    effects   = c("medianIntensityPeptide", "nPrecursors", "plate_id"),
    intercept = TRUE
)
sce_plate <- scpModelWorkflow(
    sce_plate,
    formula = ~ 1 +
        comment.X.position. +
        source.name + Diameter
)
saveRDS(sce_plate, file.path(MODELS_OUT, "sce_plate.rds"))
cat("  Saved: sce_plate.rds\n\n")

# --- Regress out full batch, refit ---
cat("[4/5] Regressing out full batch effects...\n")
sce_batch <- scpRemoveBatchEffect(
    sce,
    effects   = c("medianIntensityPeptide", "nPrecursors",
                  "plate_id", "comment.X.position."),
    intercept = TRUE
)
sce_batch <- scpModelWorkflow(
    sce_batch,
    formula = ~ 1 +
        source.name + Diameter
)
saveRDS(sce_batch, file.path(MODELS_OUT, "sce_batch.rds"))
cat("  Saved: sce_batch.rds\n\n")

# --- Pure residuals (all known effects removed) ---
cat("[5/5] Removing all known effects (residuals only)...\n")
sce_resid <- scpRemoveBatchEffect(
    sce,
    effects   = c("medianIntensityPeptide", "nPrecursors",
                  "plate_id", "comment.X.position.",
                  "source.name", "Diameter"),
    intercept = TRUE
)
saveRDS(sce_resid, file.path(MODELS_OUT, "sce_resid.rds"))
cat("  Saved: sce_resid.rds\n")

### 3.2 Feature Filtering Thresholds

Sets the n/p ratio threshold to 10 for each fitted model. Precursors below this threshold have fewer than 10 observations per model parameter and are considered unreliably fitted; they are excluded from all downstream variance analysis and dimensionality reduction. `scpModelFilterPlot` shows the distribution of n/p ratios and the fraction of features retained at the chosen threshold — inspect these to confirm the threshold is appropriate.

In [ ]:
NP_THRESHOLD <- 10

for (obj_name in c("sce", "sce_norm", "sce_plate", "sce_batch")) {
    obj <- get(obj_name)
    scpModelFilterThreshold(obj) <- NP_THRESHOLD
    assign(obj_name, obj)
}

options(repr.plot.width = 14, repr.plot.height = 8)
scpModelFilterPlot(sce)       + ggtitle("Full model")
scpModelFilterPlot(sce_norm)  + ggtitle("Post-normalisation")
scpModelFilterPlot(sce_plate) + ggtitle("Post-plate correction")
scpModelFilterPlot(sce_batch) + ggtitle("Post-batch correction")

## 4. Variance Analysis

Decomposes the variance explained by each covariate across all well-fitted precursors (n/p ≥ 10). Two complementary visualisations are produced per correction stage:

- **Average bar chart** (`scpVariancePlot`) — one bar per covariate showing the mean percentage of variance explained across all precursors. Gives a global view of which effects dominate.
- **Violin plot** — shows the full distribution of per-precursor variance fractions. A narrow violin at high values indicates that the covariate consistently explains that fraction across features; a wide violin indicates heterogeneity.

**Colour scheme** — covariates are grouped by functional role:

| Colour | Role | Variables |
|---|---|---|
| Grey | Residuals | unexplained variance |
| Pastel yellow | Normalisation | `medianIntensityPeptide`, `nPrecursors` |
| Pastel red | Batch | `plate_id`, `comment.X.position.` |
| Pastel green | Biology | `source.name`, `Diameter` |

In [ ]:
COL_RESIDUALS <- "#B0B0B0"  # grey
COL_NORM      <- "#FFE066"  # pastel yellow
COL_BATCH     <- "#FF9999"  # pastel red
COL_BIO       <- "#99CC99"  # pastel green

colour_map_full <- c(
    "Residuals"              = COL_RESIDUALS,
    "medianIntensityPeptide" = COL_NORM,
    "nPrecursors"            = COL_NORM,
    "plate_id"               = COL_BATCH,
    "comment.X.position."   = COL_BATCH,
    "source.name"            = COL_BIO,
    "Diameter"               = COL_BIO
)

colour_map_norm <- c(
    "Residuals"            = COL_RESIDUALS,
    "plate_id"             = COL_BATCH,
    "comment.X.position."  = COL_BATCH,
    "source.name"          = COL_BIO,
    "Diameter"             = COL_BIO
)

colour_map_plate <- c(
    "Residuals"            = COL_RESIDUALS,
    "comment.X.position."  = COL_BATCH,
    "source.name"          = COL_BIO,
    "Diameter"             = COL_BIO
)

colour_map_batch <- c(
    "Residuals"   = COL_RESIDUALS,
    "source.name" = COL_BIO,
    "Diameter"    = COL_BIO
)

### 4.1 Extract Variance Results

`scpVarianceAnalysis` computes the percentage of variance explained by each covariate per precursor, restricted to features passing the n/p threshold. The result is a named list of data frames, one per covariate, each containing `percentExplainedVar` values across all retained precursors.

In [ ]:
# Load from disk if models were fit in a prior session
if (!exists("sce"))       sce       <- readRDS(file.path(MODELS_OUT, "sce.rds"))
if (!exists("sce_norm"))  sce_norm  <- readRDS(file.path(MODELS_OUT, "sce_norm.rds"))
if (!exists("sce_plate")) sce_plate <- readRDS(file.path(MODELS_OUT, "sce_plate.rds"))
if (!exists("sce_batch")) sce_batch <- readRDS(file.path(MODELS_OUT, "sce_batch.rds"))

vaRes_full  <- scpVarianceAnalysis(sce)
vaRes_norm  <- scpVarianceAnalysis(sce_norm)
vaRes_plate <- scpVarianceAnalysis(sce_plate)
vaRes_batch <- scpVarianceAnalysis(sce_batch)

cat(sprintf("Precursors in variance results:\n"))
cat(sprintf("  Full model   : %d\n", nrow(vaRes_full[[1]])))
cat(sprintf("  Post-norm    : %d\n", nrow(vaRes_norm[[1]])))
cat(sprintf("  Post-plate   : %d\n", nrow(vaRes_plate[[1]])))
cat(sprintf("  Post-batch   : %d\n", nrow(vaRes_batch[[1]])))

### 4.2 Average Variance Partitioning

`scpVariancePlot` draws a single stacked bar per model showing the mean percentage of variance explained by each covariate. The custom colour scale groups covariates by functional role. Interpreting the progression across stages:

- If normalisation terms shrink substantially from the full model, they captured real intensity-scale differences.
- If biology terms grow after batch regression, biological signal was previously masked by batch noise.
- A large residual fraction at all stages suggests unmeasured sources of variation.

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5)

p_full  <- scpVariancePlot(vaRes_full)  + scale_fill_manual(values = colour_map_full)  + ggtitle("Full model")
p_norm  <- scpVariancePlot(vaRes_norm)  + scale_fill_manual(values = colour_map_norm)  + ggtitle("Post-normalisation")
p_plate <- scpVariancePlot(vaRes_plate) + scale_fill_manual(values = colour_map_plate) + ggtitle("Post-plate correction")
p_batch <- scpVariancePlot(vaRes_batch) + scale_fill_manual(values = colour_map_batch) + ggtitle("Post-batch correction")

# Save
var_fig_dir <- file.path(FIGURES_OUT, "Figure_6", "Variance")
ggsave(file.path(var_fig_dir, "variance_bar_full.png"),  p_full,  width = 10, height = 5)
ggsave(file.path(var_fig_dir, "variance_bar_norm.png"),  p_norm,  width = 10, height = 5)
ggsave(file.path(var_fig_dir, "variance_bar_plate.png"), p_plate, width = 10, height = 5)
ggsave(file.path(var_fig_dir, "variance_bar_batch.png"), p_batch, width = 10, height = 5)

print(p_full)
print(p_norm)
print(p_plate)
print(p_batch)

### 4.3 Per-feature Variance Distribution

The average bar chart can mask bimodal or heavy-tailed distributions. For each covariate, a violin plot shows the full distribution of per-precursor variance fractions. Covariates are ordered by median. The inner box shows Q1, median, Q3.

**Interpretation guide**:
- A narrow violin near 0 means the covariate explains little variance in most precursors — this is expected for well-controlled batches.
- A broad violin indicates heterogeneous covariate influence across the proteome — some features are strongly batch-affected, others not.
- Biological covariates (`source.name`) should show a long right tail after batch correction if the cell types differ substantially.

In [ ]:
plot_variance_violin <- function(vaRes,
                                 title      = "Variance explained per precursor",
                                 colour_map = NULL,
                                 width      = 8,
                                 height     = 5) {
    var_long <- lapply(names(vaRes), function(v) {
        data.frame(variable = v, pct = vaRes[[v]]$percentExplainedVar)
    }) %>% bind_rows()

    # Order by median
    var_order <- var_long %>%
        group_by(variable) %>%
        summarise(med = median(pct, na.rm = TRUE), .groups = "drop") %>%
        arrange(desc(med)) %>%
        pull(variable)

    var_long$variable <- factor(var_long$variable, levels = var_order)

    if (is.null(colour_map)) {
        n_vars     <- length(var_order)
        colour_map <- setNames(
            colorRampPalette(brewer.pal(min(n_vars, 9), "Set1"))(n_vars),
            var_order
        )
    }

    options(repr.plot.width = width, repr.plot.height = height)

    ggplot(var_long, aes(x = variable, y = pct / 100, fill = variable)) +
        geom_violin(scale = "width", alpha = 0.8, show.legend = FALSE) +
        geom_boxplot(width = 0.08, outlier.shape = NA,
                     fill = "white", show.legend = FALSE) +
        scale_fill_manual(values = colour_map) +
        scale_y_continuous(
            labels = function(x) paste0(round(x * 100), "%"),
            limits = c(0, 1)
        ) +
        theme_classic(base_size = 12) +
        labs(
            title    = title,
            subtitle = sprintf("n = %d precursors", nrow(vaRes[[1]])),
            x        = NULL,
            y        = "Fraction of variance explained"
        ) +
        theme(
            axis.text.x  = element_text(angle = 35, hjust = 1),
            plot.title   = element_text(face = "bold")
        )
}

In [ ]:
var_fig_dir <- file.path(FIGURES_OUT, "Figure_6", "Variance")

stages <- list(
    list(vaRes = vaRes_full,  title = "All effects",            cmap = colour_map_full,  fname = "variance_violin_full.png"),
    list(vaRes = vaRes_norm,  title = "Post-normalisation",     cmap = colour_map_norm,  fname = "variance_violin_norm.png"),
    list(vaRes = vaRes_plate, title = "Post-plate correction",  cmap = colour_map_plate, fname = "variance_violin_plate.png"),
    list(vaRes = vaRes_batch, title = "Post-batch correction",  cmap = colour_map_batch, fname = "variance_violin_batch.png")
)

for (s in stages) {
    p <- plot_variance_violin(s$vaRes, title = s$title, colour_map = s$cmap)
    ggsave(file.path(var_fig_dir, s$fname), p, width = 8, height = 5)
    print(p)
}

### 4.4 Variance Explained as a Function of Feature Abundance

Tests whether technical effects disproportionately dominate low-abundance features. Precursors are split into ten equal-sized bins by mean log₂ intensity across all cells, and the average variance fraction attributable to each covariate group is plotted per bin.

**Expected pattern:**
- **Normalisation and batch effects** should explain a *decreasing* fraction of variance as abundance increases — low-abundance features are noisier and more susceptible to stochastic technical variation.
- **Biological variance** (`source.name`, `Diameter`) should be relatively stable across bins, or increase slightly at high abundance where features are reliably quantified in both cell types.
- If batch variance remains high even at high abundance, the batch effects are systematic (e.g. plate-wide intensity offsets) rather than noise-driven; these are best addressed by explicit modelling rather than feature filtering.

The analysis is performed on the full model since it contains variance estimates for all covariate groups simultaneously.

In [ ]:
if (!exists("sce"))      sce      <- readRDS(file.path(MODELS_OUT, "sce.rds"))
if (!exists("vaRes_full")) vaRes_full <- scpVarianceAnalysis(sce)

# Mean log2-intensity per precursor (proxy for abundance)
log_mat   <- assay(sce)
feat_mean <- rowMeans(log_mat, na.rm = TRUE)

# Restrict to features present in the variance results
feat_in_vaRes <- intersect(names(feat_mean), rownames(vaRes_full[[1]]))
cat(sprintf("Features used: %d\n", length(feat_in_vaRes)))

# Build long data frame: feature × variable × pct, plus mean intensity
n_bins <- 10
var_abund <- lapply(names(vaRes_full), function(v) {
    df <- vaRes_full[[v]][feat_in_vaRes, ]
    data.frame(
        feature  = feat_in_vaRes,
        variable = v,
        pct      = df$percentExplainedVar,
        mean_int = feat_mean[feat_in_vaRes]
    )
}) %>% bind_rows()

var_abund$abundance_bin <- cut(
    var_abund$mean_int,
    breaks         = quantile(unique(var_abund$mean_int),
                              probs = seq(0, 1, 1 / n_bins), na.rm = TRUE),
    labels         = paste0("D", seq_len(n_bins)),
    include.lowest = TRUE
)

# Average pct per (abundance_bin, variable)
var_abund_summary <- var_abund %>%
    group_by(abundance_bin, variable) %>%
    summarise(mean_pct = mean(pct, na.rm = TRUE), .groups = "drop")

group_colours <- c(
    "Residuals"              = COL_RESIDUALS,
    "medianIntensityPeptide" = COL_NORM,
    "nPrecursors"            = COL_NORM,
    "plate_id"               = COL_BATCH,
    "comment.X.position."   = COL_BATCH,
    "source.name"            = COL_BIO,
    "Diameter"               = COL_BIO
)

options(repr.plot.width = 12, repr.plot.height = 5)

p_abund <- ggplot(var_abund_summary,
                  aes(x = abundance_bin, y = mean_pct / 100,
                      colour = variable, group = variable)) +
    geom_line(linewidth = 0.9) +
    geom_point(size = 2.5) +
    scale_colour_manual(values = group_colours) +
    scale_y_continuous(labels = scales::percent_format(accuracy = 1), limits = c(0, NA)) +
    theme_classic(base_size = 12) +
    labs(
        title    = "Variance explained by abundance decile (full model)",
        subtitle = sprintf("n = %d precursors across %d bins", length(feat_in_vaRes), n_bins),
        x        = "Abundance decile  (D1 = lowest intensity)",
        y        = "Mean fraction of variance explained",
        colour   = "Covariate"
    ) +
    theme(axis.text.x = element_text(angle = 30, hjust = 1))

ggsave(file.path(FIGURES_OUT, "Figure_6", "Variance", "variance_by_abundance.png"),
       p_abund, width = 12, height = 5)
print(p_abund)

## 5. Dimensionality Reduction

Uses `omicsGMF` to perform probabilistic matrix factorisation (GMF) that simultaneously imputes missing values during decomposition. GMF is preferable to PCA here because single-cell proteomics data is highly sparse and PCA cannot handle missingness without prior imputation.

Three steps are performed:

1. **Rank selection** — `runRankGMF` fits models at increasing rank (1–10) and plots reconstruction error, helping to identify the elbow that indicates the intrinsic dimensionality of the data.
2. **GMF fitting** — `runGMF` is applied at the chosen rank (4 components) to each of the four SCE objects, capturing the latent structure at each correction stage.
3. **Visualisation** — `plotReducedDim` colours the GMF embedding by key metadata dimensions to assess which sources of variation remain at each stage.

> **Runtime note** — `runRankGMF` takes ~45 minutes; each `runGMF` call takes ~30–60 minutes. Objects are saved to disk after fitting.

### 5.1 Rank Selection

`runRankGMF` fits GMF models from rank 1 to `maxcomp` and returns an object that can be passed to `plotRank`. The elbow in the reconstruction error curve indicates where adding more components yields diminishing returns — this is the rank used for the full GMF fit below. The n/p filter is applied first to restrict to reliably fitted precursors.

In [ ]:
if (!exists("sce")) sce <- readRDS(file.path(MODELS_OUT, "sce.rds"))

np_ratio    <- scpModelFilterNPRatio(sce, filtered = FALSE)
sce_filtered <- sce[np_ratio > NP_THRESHOLD, ]
cat(sprintf("Precursors after n/p filter: %d / %d\n",
            nrow(sce_filtered), length(np_ratio)))

# ~45 minutes
sce_rankGMF <- runRankGMF(
    sce_filtered,
    exprs_values = "raw",
    family       = gaussian(),
    maxcomp      = 10,
    BPPARAM      = MulticoreParam(workers = 64)
)

options(repr.plot.width = 8, repr.plot.height = 5)
plotRank(sce_rankGMF, maxcomp = 10)

### 5.2 Run GMF

Fits a 4-component GMF to each correction stage. The `ntop = 5000` parameter (for the corrected-stage models) restricts the feature set to the 5,000 most variable precursors, substantially reducing runtime without sacrificing the major axes of variation. The full `sce` is run without `ntop` to capture the complete latent structure of the raw data.

**Parameters used:**

| Parameter | Value | Meaning |
|---|---|---|
| `ncomponents` | `4` | Number of latent dimensions (from rank plot) |
| `family` | `gaussian()` | Gaussian noise model (appropriate for log-transformed data) |
| `ntop` | `5000` (corrected stages) | Feature selection for speed |
| `exprs_values` | `"raw"` | Use the residualised intensity matrix |

In [ ]:
N_COMPONENTS <- 4

# Full SCE (no ntop — captures complete structure)
cat("[1/4] Running GMF on full SCE...\n")
sce_GMF <- runGMF(
    sce_filtered,
    exprs_values = "raw",
    family       = gaussian(),
    ncomponents  = N_COMPONENTS,
    name         = "GMF",
    BPPARAM      = MulticoreParam(workers = 64)
)
saveRDS(sce_GMF, file.path(MODELS_OUT, "sce_GMF.rds"))
cat("  Saved: sce_GMF.rds\n\n")

# Corrected stages — use ntop = 5000 for speed
if (!exists("sce_norm"))  sce_norm  <- readRDS(file.path(MODELS_OUT, "sce_norm.rds"))
if (!exists("sce_plate")) sce_plate <- readRDS(file.path(MODELS_OUT, "sce_plate.rds"))
if (!exists("sce_batch")) sce_batch <- readRDS(file.path(MODELS_OUT, "sce_batch.rds"))

# Rename assay for corrected objects so runGMF can find it
assayNames(sce_norm)[1]  <- "raw"
assayNames(sce_plate)[1] <- "raw"
assayNames(sce_batch)[1] <- "raw"

for (stage in list(
    list(obj = sce_norm,  name = "sce_norm",  label = "[2/4] Post-normalisation"),
    list(obj = sce_plate, name = "sce_plate", label = "[3/4] Post-plate"),
    list(obj = sce_batch, name = "sce_batch", label = "[4/4] Post-batch")
)) {
    cat(sprintf("%s...\n", stage$label))
    np   <- scpModelFilterNPRatio(stage$obj, filtered = FALSE)
    filt <- stage$obj[np > NP_THRESHOLD, ]
    gmf  <- runGMF(
        filt,
        ntop         = 5000,
        exprs_values = "raw",
        family       = gaussian(),
        ncomponents  = N_COMPONENTS,
        name         = "GMF",
        BPPARAM      = MulticoreParam(workers = 64)
    )
    out_name <- paste0(stage$name, "_GMF")
    assign(out_name, gmf)
    saveRDS(gmf, file.path(MODELS_OUT, paste0(out_name, ".rds")))
    cat(sprintf("  Saved: %s.rds\n\n", out_name))
}

### 5.3 Plot Reduced Dimensions

Colours the first two GMF components by key metadata. The expected pattern after successive correction:

- **Full SCE** (`sce_GMF`): cells cluster by `plate_id` and `source.name` simultaneously — both batch and biology visible.
- **Post-normalisation** (`sce_norm_GMF`): intensity-scale batch reduced; `plate_id` clusters may persist.
- **Post-plate** (`sce_plate_GMF`): plate clustering dissolved; `comment.X.position.` may still separate.
- **Post-batch** (`sce_batch_GMF`): cells should separate primarily by `source.name` (biology), with `plate_id` clusters absent.

In [ ]:
if (!exists("sce_GMF"))       sce_GMF       <- readRDS(file.path(MODELS_OUT, "sce_GMF.rds"))
if (!exists("sce_norm_GMF"))  sce_norm_GMF  <- readRDS(file.path(MODELS_OUT, "sce_norm_GMF.rds"))
if (!exists("sce_plate_GMF")) sce_plate_GMF <- readRDS(file.path(MODELS_OUT, "sce_plate_GMF.rds"))
if (!exists("sce_batch_GMF")) sce_batch_GMF <- readRDS(file.path(MODELS_OUT, "sce_batch_GMF.rds"))

colour_dims <- c("plate_id", "source.name", "comment.X.position.",
                 "n_precursors", "medianIntensityPeptide", "Reinjection")

options(repr.plot.width = 7, repr.plot.height = 6)

for (stage in list(
    list(obj = sce_GMF,       label = "Full model"),
    list(obj = sce_norm_GMF,  label = "Post-normalisation"),
    list(obj = sce_plate_GMF, label = "Post-plate correction"),
    list(obj = sce_batch_GMF, label = "Post-batch correction")
)) {
    cat(sprintf("--- %s ---\n", stage$label))
    for (col_by in colour_dims) {
        tryCatch(
            print(
                plotReducedDim(stage$obj, dimred = "GMF", colour_by = col_by) +
                    ggtitle(sprintf("%s | colour: %s", stage$label, col_by))
            ),
            error = function(e) message(sprintf("  Skipping %s: %s", col_by, e$message))
        )
    }
}

### 5.4 Batch Mixing and Biological Separation

Quantifies the quality of each correction stage numerically using average silhouette widths computed on the first two GMF components. Two complementary scores are reported per stage:

| Metric | Variable scored | Interpretation |
|---|---|---|
| **Biological separation** | `source.name` | Higher → cell types form more distinct clusters in embedding space |
| **Batch structure** | `plate_id` | Lower (towards −1) → cells are well-mixed across plates; no plate clustering |

A successful batch correction should show **increasing** biological silhouette and **decreasing** batch silhouette as correction progresses. If both decrease together, the correction is over-fitting and eroding biological structure alongside batch effects.

> Silhouette scores are computed on the first two GMF dimensions for speed; using all four components may give slightly different absolute values but the relative ordering across stages should be stable.

In [ ]:
library(cluster)

compute_silhouettes <- function(gmf_obj, n_dims = 2,
                                batch_var = "plate_id",
                                bio_var   = "source.name") {
    coords    <- reducedDim(gmf_obj, "GMF")[, seq_len(n_dims)]
    cd        <- colData(gmf_obj)
    d_mat     <- dist(coords)
    bio_sil   <- mean(silhouette(as.integer(factor(cd[[bio_var]])),   d_mat)[, 3])
    batch_sil <- mean(silhouette(as.integer(factor(cd[[batch_var]])), d_mat)[, 3])
    data.frame(bio_silhouette = bio_sil, batch_silhouette = batch_sil)
}

if (!exists("sce_GMF"))       sce_GMF       <- readRDS(file.path(MODELS_OUT, "sce_GMF.rds"))
if (!exists("sce_norm_GMF"))  sce_norm_GMF  <- readRDS(file.path(MODELS_OUT, "sce_norm_GMF.rds"))
if (!exists("sce_plate_GMF")) sce_plate_GMF <- readRDS(file.path(MODELS_OUT, "sce_plate_GMF.rds"))
if (!exists("sce_batch_GMF")) sce_batch_GMF <- readRDS(file.path(MODELS_OUT, "sce_batch_GMF.rds"))

stage_list <- list(
    "Full model"         = sce_GMF,
    "Post-normalisation" = sce_norm_GMF,
    "Post-plate"         = sce_plate_GMF,
    "Post-batch"         = sce_batch_GMF
)

sil_results <- lapply(names(stage_list), function(nm) {
    cat(sprintf("  Computing silhouettes: %s...\n", nm))
    res       <- compute_silhouettes(stage_list[[nm]])
    res$stage <- nm
    res
}) %>% bind_rows()

sil_results$stage <- factor(sil_results$stage, levels = names(stage_list))

cat("\nSilhouette scores per correction stage:\n")
print(sil_results[, c("stage", "bio_silhouette", "batch_silhouette")])

# Reshape to long format without tidyr
sil_long <- rbind(
    data.frame(stage      = sil_results$stage,
               metric     = "Biological separation (source.name)",
               silhouette = sil_results$bio_silhouette),
    data.frame(stage      = sil_results$stage,
               metric     = "Batch structure (plate_id)",
               silhouette = sil_results$batch_silhouette)
)
sil_long$stage <- factor(sil_long$stage, levels = names(stage_list))

options(repr.plot.width = 9, repr.plot.height = 5)

p_sil <- ggplot(sil_long, aes(x = stage, y = silhouette, fill = metric)) +
    geom_col(position = position_dodge(width = 0.7), width = 0.6) +
    geom_hline(yintercept = 0, linetype = "dashed", colour = "grey40") +
    scale_fill_manual(values = c(
        "Biological separation (source.name)" = COL_BIO,
        "Batch structure (plate_id)"          = COL_BATCH
    )) +
    theme_classic(base_size = 12) +
    labs(
        title    = "Silhouette scores across correction stages",
        subtitle = "First 2 GMF components; higher bio + lower batch = better correction",
        x        = NULL,
        y        = "Mean silhouette width",
        fill     = NULL
    ) +
    theme(
        axis.text.x     = element_text(angle = 25, hjust = 1),
        legend.position = "top"
    )

ggsave(file.path(FIGURES_OUT, "Figure_6", "Variance", "silhouette_scores.png"),
       p_sil, width = 9, height = 5)
print(p_sil)

## 6. Differential Expression Analysis

Tests for differential abundance between K562 and HEK293T cells at each correction stage using `scpDifferentialAnalysis`, which applies a moderated t-test per precursor via the fitted scplainer model. Results are visualised as volcano plots.

**Interpreting the correction-stage comparison:**
- More significant hits after batch correction indicate that batch noise was suppressing differential signal.
- Similar hit lists across stages support that the biological contrast is robust to technical variation.
- Hits appearing only after batch correction should be treated cautiously — they may be regression artefacts.

In [ ]:
if (!exists("sce_norm"))  sce_norm  <- readRDS(file.path(MODELS_OUT, "sce_norm.rds"))
if (!exists("sce_plate")) sce_plate <- readRDS(file.path(MODELS_OUT, "sce_plate.rds"))
if (!exists("sce_batch")) sce_batch <- readRDS(file.path(MODELS_OUT, "sce_batch.rds"))

contrast <- list(c("source.name", "HEK293T", "K562"))

options(repr.plot.width = 12, repr.plot.height = 5)

for (stage in list(
    list(obj = sce_norm,  label = "Post-normalisation"),
    list(obj = sce_plate, label = "Post-plate correction"),
    list(obj = sce_batch, label = "Post-batch correction")
)) {
    res <- scpDifferentialAnalysis(stage$obj, contrasts = contrast)
    p   <- scpVolcanoPlot(res) |> wrap_plots(guides = "collect")
    print(p + plot_annotation(title = sprintf("Differential analysis: %s", stage$label)))
}

## 7. Export

Converts the batch-corrected SCE (`sce_batch`) to AnnData format for downstream Python-based analyses. The model metadata is stripped before conversion to avoid serialisation issues. The exported file is written to `MODELS_OUT/sce_batch.h5ad` and can be read directly with `anndata.read_h5ad()` in Python.

In [ ]:
if (!exists("sce_batch")) sce_batch <- readRDS(file.path(MODELS_OUT, "sce_batch.rds"))

# Rename assay and strip model metadata for clean export
assayNames(sce_batch)[1] <- "raw"
metadata(sce_batch)      <- list()

adata <- SCE2AnnData(sce_batch)

out_path <- file.path(MODELS_OUT, "sce_batch.h5ad")
adata$write_h5ad(out_path)
cat(sprintf("Exported to: %s\n", out_path))